# Laboratorio práctico de ETL: Kaismart Solutions S.A.S.

Notebook de presentación del laboratorio. Usa los mismos módulos de `src/` que ejecutan `main.py` y `orchestrator.py`.
Los autores están en `config/config.yaml` y el informe completo en `docs/Informe_KaismartSolutions.html`.

In [1]:
import os, sys
import yaml
import pandas as pd

# El notebook vive en notebooks/: se trabaja desde la raíz del proyecto
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

with open("config/config.yaml", encoding="utf-8") as f:
    print("Autores:", ", ".join(yaml.safe_load(f)["autores"]))

Autores: Miguel Ángel Méndez Rodríguez, Maicol Andrés Narváez Rincón, Karin Stephany Parra Rosero, Angie Tatiana Rodríguez Duque


## Parte 1. Extracción desde la base de datos MySQL

Se usa `mysql.connector`. Las credenciales se leen del archivo `.env`, se consulta la tabla `ventas` completa y la conexión se cierra en el bloque `finally`.

In [2]:
from src.extract.extract_db import extraer_ventas_db
df_ventas = extraer_ventas_db()

[2026-10-05 20:30:12,063] [INFO] - PARTE 1: Iniciando extracción desde la Base de Datos MySQL...


[2026-10-05 20:30:12,764] [INFO] - Conexión MySQL establecida.


[2026-10-05 20:30:20,762] [INFO] - df_ventas extraído con éxito. Registros: 5000, Columnas: 17


[2026-10-05 20:30:20,891] [INFO] - Conexión MySQL cerrada correctamente.



  PARTE 1: COMPROBACIÓN DE EXTRACCIÓN - df_ventas
df_ventas.shape: (5000, 17)
df_ventas.columns:
['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal', 'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'medio_pago', 'calificacion_cliente']

df_ventas.head():
   id_venta  pedido_id         fecha_venta id_cliente       ciudad             canal id_tienda          categoria               producto  cantidad  precio_unitario  descuento_pct  valor_bruto  \
0         1  PED000001 2026-01-18 22:06:15   CLI00011  Bogotá D.C.  Aplicación móvil       NaN           Deportes       Banda caminadora         1        2199900.0           10.0    2199900.0   
1         2  PED000002 2026-04-25 21:07:01   CLI00802         Cali        Página web       NaN           Deportes  Mancuernas ajustables         1         399900.0           10.0     399900.0   
2         3  PED000003 2026-03-28 02:39:43   CLI0025

In [3]:
df_ventas.shape

(5000, 17)

In [4]:
df_ventas.columns

Index(['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal', 'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento',
       'valor_neto', 'medio_pago', 'calificacion_cliente'],
      dtype='str')

In [5]:
df_ventas.head()

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.0,10.0,2199900.0,219990.0,1979910.0,Tarjeta crédito,NaN
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.0,10.0,399900.0,39990.0,359910.0,Tarjeta crédito,NaN
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.0,10.0,899900.0,89990.0,809910.0,Tarjeta crédito,4.0
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.0,15.0,944800.0,141720.0,803080.0,PSE,5.0
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.0,0.0,119900.0,0.0,119900.0,Tarjeta crédito,NaN


In [6]:
df_ventas.sample(5, random_state=42)

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente
1501,1502,PED001502,2026-06-10 08:36:33,CLI02390,Bucaramanga,Tienda física,BGA-01,Tecnología,Monitor 24 pulgadas,2,713900.0,5.0,1427800.0,71390.0,1356410.0,Tarjeta débito,NaN
2586,2587,PED002587,2026-05-09 09:23:12,CLI01262,Bucaramanga,Aplicación móvil,NaN,Tecnología,Audífonos Bluetooth,2,149900.0,5.0,299800.0,14990.0,284810.0,Billetera digital,4.0
2653,2654,PED002654,2026-05-20 23:03:48,CLI01371,Cali,Aplicación móvil,NaN,Oficina,Base para portátil,1,112100.0,0.0,112100.0,0.0,112100.0,Tarjeta crédito,NaN
1055,1056,PED001056,2026-03-18 19:24:39,CLI00437,Medellín,Tienda física,MDE-01,Deportes,Bicicleta estática,2,1499900.0,0.0,2999800.0,0.0,2999800.0,Efectivo,5.0
705,706,PED000706,2026-06-06 19:01:06,CLI01135,Bogotá D.C.,Página web,NaN,Tecnología,Teclado mecánico,1,229900.0,5.0,229900.0,11495.0,218405.0,PSE,NaN


## Parte 2. Extracción desde el archivo Excel

In [7]:
from src.extract.extract_excel import extraer_logistica_excel
df_logistica = extraer_logistica_excel()

[2026-10-05 20:30:20,955] [INFO] - Iniciando PARTE 2: Extracción desde Excel (Sistema Logístico)...


[2026-10-05 20:30:25,715] [INFO] - Extracción exitosa de df_logistica con forma (50000, 14)



    COMPROBACIÓN EXTRACCIÓN: df_logistica
df_logistica.shape: (50000, 14)

df_logistica.columns:
['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia', 'fecha_prometida_entrega', 'tiempo_etapa_horas', 'costo_envio', 'incidencia', 'observacion', 'Unnamed: 13']

df_logistica.head():
   evento_id  pedido_id         fecha_evento              estado_evento centro_logistico ciudad_destino transportadora numero_guia fecha_prometida_entrega  tiempo_etapa_horas  costo_envio  \
0          1  PED000001  2026-01-18 22:06:15            Pedido recibido      CEDI Bogotá    Bogotá D.C.            NaN         NaN     2026-01-20 14:06:15                 NaN         8900   
1          2  PED000001  2026-01-18 22:36:53              Pago aprobado      CEDI Bogotá    Bogotá D.C.            NaN         NaN     2026-01-20 14:06:15                0.51         8900   
2          3  PED000001  2026-01-18 23:31:22  Orden enviada a logíst

In [8]:
df_logistica.shape

(50000, 14)

In [9]:
df_logistica.columns

Index(['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia', 'fecha_prometida_entrega', 'tiempo_etapa_horas',
       'costo_envio', 'incidencia', 'observacion', 'Unnamed: 13'],
      dtype='str')

In [10]:
df_logistica.head()

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
0,1,PED000001,2026-01-18 22:06:15,Pedido recibido,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,NaN,8900,NaN,NaN,"=AI("""")"
1,2,PED000001,2026-01-18 22:36:53,Pago aprobado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.51,8900,NaN,NaN,NaN
2,3,PED000001,2026-01-18 23:31:22,Orden enviada a logística,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,0.91,8900,NaN,NaN,NaN
3,4,PED000001,2026-01-19 02:03:55,En alistamiento,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,2.54,8900,NaN,NaN,NaN
4,5,PED000001,2026-01-19 06:14:51,Empacado,CEDI Bogotá,Bogotá D.C.,NaN,NaN,2026-01-20 14:06:15,4.18,8900,NaN,NaN,NaN


In [11]:
df_logistica.sample(5, random_state=42)

,evento_id,pedido_id,fecha_evento,estado_evento,centro_logistico,ciudad_destino,transportadora,numero_guia,fecha_prometida_entrega,tiempo_etapa_horas,costo_envio,incidencia,observacion,Unnamed: 13
33553,33554,PED003356,2026-04-04 09:51:57,En alistamiento,CEDI Pereira,Pereira,NaN,NaN,2026-04-06 02:08:22,1.78,9900,NaN,Pedido procesado dentro de la ventana prevista.,NaN
9427,9428,PED000943,2026-04-14 11:09:42,En tránsito,CEDI Bucaramanga,Bucaramanga,Servientrega,KMS77388424,2026-04-15 16:24:14,9.56,8900,NaN,NaN,NaN
199,200,PED000020,2026-02-08 21:23:51,Entregado,CEDI Bogotá,Bogotá D.C.,TCC,KMS46254024,2026-02-09 11:26:46,5.13,13900,NaN,NaN,NaN
12447,12448,PED001245,2026-05-02 00:24:54,En tránsito,CEDI Cali,Cali,Inter Rapidísimo,KMS96024511,2026-05-03 00:03:57,9.38,15900,NaN,NaN,NaN
39489,39490,PED003949,2026-06-03 14:44:47,Entregado,CEDI Bogotá,Bogotá D.C.,Envia,KMS90495560,2026-06-03 02:22:34,9.09,17490,NaN,NaN,NaN


## Capa Bronze

Apenas se extraen, los datos originales se guardan sin cambios en `data/bronze/`. También queda una copia por carga en `data/bronze/historico/AAAAMMDD_HHMMSS/`, con el `.xlsx` original. Se abre el manifiesto de la ejecución. Las partes 3 a 6 trabajan sobre estos mismos datos.

In [12]:
from src.pipeline import aterrizar_bronze
manifiesto = aterrizar_bronze(df_ventas, df_logistica)
print("Carga de Bronze:", manifiesto.id_carga)

[2026-10-05 20:30:25,830] [INFO] - Guardado data\bronze\ventas_bronze.parquet (5000 filas x 17 columnas)


[2026-10-05 20:30:25,887] [INFO] - Guardado data\bronze\logistica_bronze.parquet (50000 filas x 14 columnas)


[2026-10-05 20:30:25,904] [INFO] - Guardado data\bronze\historico\20261005_203025\ventas_bronze.parquet (5000 filas x 17 columnas)


[2026-10-05 20:30:25,958] [INFO] - Guardado data\bronze\historico\20261005_203025\logistica_bronze.parquet (50000 filas x 14 columnas)


[2026-10-05 20:30:25,967] [INFO] - [BRONZE] Carga 20261005_203025: datos originales respaldados sin cambios.


Carga de Bronze: 20261005_203025


## Parte 3. Comprensión inicial de los datasets

In [13]:
from src.eda import eda_inspection as eda
eda.comprension_inicial(df_ventas, df_logistica)


############################################################
 PARTE 3: COMPRENSIÓN INICIAL DE LOS DATASETS
############################################################

---> Exploración de df_ventas <---
1. Registros (filas): 5000
2. Variables (columnas): 17
3. Nombres de variables: ['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal', 'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'medio_pago', 'calificacion_cliente']

4 y 5. Tipos de datos y registros no nulos:
            Variable   Tipo de dato  Registros no nulos      Clasificación
            id_venta          int64                5000      Identificador
           pedido_id            str                5000      Identificador
         fecha_venta datetime64[us]                5000              Fecha
          id_cliente            str                5000      Identificador
              ciudad            str              

   - valor_bruto = precio_unitario × cantidad: 5.000 de 5.000 cumplen (0 inconsistentes)
   - valor_descuento = valor_bruto × descuento_pct / 100: 5.000 de 5.000 cumplen (0 inconsistentes)
   - valor_neto = valor_bruto − valor_descuento: 5.000 de 5.000 cumplen (0 inconsistentes)
   - Valores negativos en variables numéricas: 0
   - Integridad referencial por pedido_id: 0 ventas sin eventos logísticos; 0 pedidos logísticos sin venta
   - Ciudad de la venta vs ciudad_destino logística: 0 pedidos con ciudad distinta
   - Eventos por pedido: mín 8, máx 12, moda 10
   - Pedidos que no pasan por los 10 estados: 98 pedidos con trazabilidad incompleta
   - costo_envio constante dentro de cada pedido: 5.000 de 5.000 pedidos (se repite en cada evento: no sumar por evento)
   - numero_guia compartido por pedidos distintos: 1: KMS64861259
   - Fechas de logística almacenadas como texto: fecha_evento y fecha_prometida_entrega son str; 0 valores no convertibles a fecha
   - Columna residual del Exce

In [14]:
df_ventas.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_venta              5000 non-null   int64         
 1   pedido_id             5000 non-null   str           
 2   fecha_venta           5000 non-null   datetime64[us]
 3   id_cliente            5000 non-null   str           
 4   ciudad                5000 non-null   str           
 5   canal                 5000 non-null   str           
 6   id_tienda             1499 non-null   str           
 7   categoria             5000 non-null   str           
 8   producto              5000 non-null   str           
 9   cantidad              5000 non-null   int64         
 10  precio_unitario       5000 non-null   float64       
 11  descuento_pct         5000 non-null   float64       
 12  valor_bruto           5000 non-null   float64       
 13  valor_descuento       5000 no

In [15]:
df_logistica.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   evento_id                50000 non-null  int64  
 1   pedido_id                50000 non-null  str    
 2   fecha_evento             49935 non-null  str    
 3   estado_evento            50000 non-null  str    
 4   centro_logistico         49910 non-null  str    
 5   ciudad_destino           49955 non-null  str    
 6   transportadora           19912 non-null  str    
 7   numero_guia              19932 non-null  str    
 8   fecha_prometida_entrega  49945 non-null  str    
 9   tiempo_etapa_horas       45000 non-null  float64
 10  costo_envio              50000 non-null  int64  
 11  incidencia               511 non-null    str    
 12  observacion              1723 non-null   str    
 13  Unnamed: 13              1 non-null      str    
dtypes: float64(1), int64(2), str(11)


## Parte 4. Perfil inicial de calidad del dato

En esta parte no se elimina ni se imputa ningún valor.

### 4.1 Valores nulos

In [16]:
eda.tabla_nulos(df_ventas)

,Variable,Cantidad nulos,Porcentaje (%)
0,id_tienda,3501,70.02
1,calificacion_cliente,2172,43.44
2,canal,0,0.00
3,cantidad,0,0.00
4,categoria,0,0.00
5,ciudad,0,0.00
6,descuento_pct,0,0.00
7,fecha_venta,0,0.00
8,id_cliente,0,0.00
9,id_venta,0,0.00


In [17]:
eda.tabla_nulos(df_logistica)

,Variable,Cantidad nulos,Porcentaje (%)
0,Unnamed: 13,49999,99.998
1,incidencia,49489,98.978
2,observacion,48277,96.554
3,transportadora,30088,60.176
4,numero_guia,30068,60.136
5,tiempo_etapa_horas,5000,10.000
6,centro_logistico,90,0.180
7,fecha_evento,65,0.130
8,fecha_prometida_entrega,55,0.110
9,ciudad_destino,45,0.090


In [18]:
eda.diagnostico_nulos(df_ventas, df_logistica)

,DataFrame,Variable,Tipo de nulo,Interpretación
0,df_ventas,id_tienda,Normal,Es nulo en 3.501 de 3.501 ventas web/app y en ...
1,df_ventas,calificacion_cliente,Normal,Calificar es opcional para el cliente. Debe te...
2,df_logistica,transportadora / numero_guia,Normal antes del despacho,Nulos antes del despacho: 30.008 / 30.008 (el ...
3,df_logistica,transportadora / numero_guia,Problema,Nulos en estados de transporte (Despachado en ...
4,df_logistica,tiempo_etapa_horas,Normal,Los 5.000 nulos están en: Pedido recibido (5.0...
5,df_logistica,incidencia,Normal,Solo se registra cuando hay una novedad.
6,df_logistica,observacion,Normal,Solo se registra cuando hay algo que anotar (1...
7,df_logistica,"centro_logistico, ciudad_destino, fecha_promet...",Problema,Son datos del pedido y deben existir en todos ...
8,df_logistica,fecha_evento,Problema,Todo evento ocurrido debe tener fecha y hora.
9,df_logistica,Unnamed: 13,Problema (columna residual),49.999 nulos de 50.000: columna sin nombre que...


### 4.2 Valores únicos (`nunique`)

In [19]:
eda.tabla_cardinalidad(df_ventas)

,Variable,Clasificación,Valores únicos,% sobre registros,Evaluación
0,id_venta,Identificador,5000,100.00,Identificador único
1,pedido_id,Identificador,5000,100.00,Identificador único
2,fecha_venta,Fecha,5000,100.00,Fecha (casi única por registro)
3,id_cliente,Identificador,2292,45.84,Identificador con repetidos (alta cardinalidad)
4,valor_neto,Numérica,1325,26.50,Numérica continua
5,valor_descuento,Numérica,693,13.86,Numérica continua
6,valor_bruto,Numérica,448,8.96,Numérica continua
7,precio_unitario,Numérica,133,2.66,Numérica continua
8,producto,Categórica,30,0.60,Cardinalidad media
9,id_tienda,Identificador,10,0.20,"Clave que se repite (~149,9 filas por valor)"


In [20]:
eda.tabla_cardinalidad(df_logistica)

,Variable,Clasificación,Valores únicos,% sobre registros,Evaluación
0,evento_id,Identificador,49900,99.800,Identificador con repetidos (alta cardinalidad)
1,fecha_evento,Fecha,49765,99.530,Fecha (casi única por registro)
2,pedido_id,Identificador,5000,10.000,"Clave que se repite (~10,0 filas por valor)"
3,numero_guia,Identificador,4999,9.998,"Clave que se repite (~4,0 filas por valor)"
4,fecha_prometida_entrega,Fecha,4999,9.998,Fecha que se repite
5,tiempo_etapa_horas,Tiempo (duración numérica),1748,3.496,Numérica continua
6,costo_envio,Numérica,12,0.024,Numérica discreta (pocos valores)
7,estado_evento,Categórica,10,0.020,Baja cardinalidad (pocos valores posibles)
8,observacion,Categórica,9,0.018,Baja cardinalidad (pocos valores posibles)
9,centro_logistico,Categórica,6,0.012,Baja cardinalidad (pocos valores posibles)


### 4.3 Duplicados

In [21]:
eda.analizar_duplicados(df_ventas, df_logistica)

{'ventas_filas_duplicadas': 0,
 'ventas_id_venta_repetidos': 0,
 'ventas_pedido_id_repetidos': 0,
 'logistica_filas_duplicadas': 99,
 'logistica_evento_id_repetidos': 100,
 'logistica_evento_id_repetidos_por_fila_identica': 99,
 'logistica_evento_id_repetidos_con_contenido_distinto': 1,
 'logistica_pedido_id_repetidos': 45000,
 'logistica_guias_compartidas': ['KMS64861259']}

### 4.4 Tipos de datos

In [22]:
eda.tabla_tipos(df_ventas)

,Variable,Tipo detectado,Diagnóstico
0,id_venta,int64,OK: clave técnica entera (no se opera aritméti...
1,pedido_id,str,OK (texto)
2,fecha_venta,datetime64[us],OK (datetime)
3,id_cliente,str,OK (texto)
4,ciudad,str,OK (texto)
5,canal,str,OK (texto)
6,id_tienda,str,OK (texto)
7,categoria,str,OK (texto)
8,producto,str,OK (texto)
9,cantidad,int64,OK (numérica)


In [23]:
eda.tabla_tipos(df_logistica)

,Variable,Tipo detectado,Diagnóstico
0,evento_id,int64,OK: clave técnica entera (no se opera aritméti...
1,pedido_id,str,OK (texto)
2,fecha_evento,str,"REVISAR: fecha almacenada como str, convertir ..."
3,estado_evento,str,OK (texto)
4,centro_logistico,str,OK (texto)
5,ciudad_destino,str,OK (texto)
6,transportadora,str,OK (texto)
7,numero_guia,str,OK (texto)
8,fecha_prometida_entrega,str,"REVISAR: fecha almacenada como str, convertir ..."
9,tiempo_etapa_horas,float64,"OK: duración en horas (numérica), no es una fecha"


## Parte 5. Estadísticos descriptivos y resúmenes

In [24]:
df_ventas.describe()

,id_venta,fecha_venta,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,calificacion_cliente
count,5000.000000,5000,5000.000000,5.000000e+03,5000.000000,5.000000e+03,5.000000e+03,5.000000e+03,2828.000000
mean,2500.500000,2026-04-02 02:17:44.840000,1.456200,4.559502e+05,5.276000,6.682881e+05,3.459310e+04,6.336950e+05,4.094767
min,1.000000,2026-01-01 09:08:59,1.000000,7.590000e+04,0.000000,7.590000e+04,0.000000e+00,6.392000e+04,1.000000
25%,1250.750000,2026-02-15 18:42:38.750000,1.000000,1.424000e+05,0.000000,1.888000e+05,0.000000e+00,1.763100e+05,4.000000
50%,2500.500000,2026-04-02 06:39:45,1.000000,2.899000e+05,5.000000,3.918000e+05,5.495000e+03,3.671100e+05,4.000000
75%,3750.250000,2026-05-17 16:01:02.750000,2.000000,6.859000e+05,10.000000,8.074000e+05,3.597000e+04,7.649150e+05,5.000000
max,5000.000000,2026-06-30 20:42:35,4.000000,2.309900e+06,20.000000,8.799600e+06,1.671920e+06,8.799600e+06,5.000000
std,1443.520003,NaN,0.743903,4.458085e+05,6.124649,8.289038e+05,7.901831e+04,7.880753e+05,1.013632


In [25]:
df_logistica.describe()

,evento_id,tiempo_etapa_horas,costo_envio
count,50000.000000,45000.000000,50000.000000
mean,24996.359960,4.004721,13609.494400
std,14436.332192,3.460415,3607.858756
min,1.000000,0.080000,8900.000000
25%,12494.750000,1.250000,9900.000000
50%,24987.500000,3.055000,13900.000000
75%,37501.250000,5.890000,15900.000000
max,50000.000000,24.690000,20790.000000


In [26]:
eda.resumen_categoricas(df_ventas)

,Variable,Valores únicos,Categoría más frecuente,Frecuencia,% de no nulos,Categorías
0,ciudad,6,Bogotá D.C.,1653,33.060000,"Bogotá D.C., Cali, Medellín, Barranquilla, Buc..."
1,canal,3,Página web,2020,40.400000,"Página web, Tienda física, Aplicación móvil"
2,categoria,5,Tecnología,1366,27.320000,"Tecnología, Electrodomésticos, Hogar, Deportes..."
3,producto,30,Monitor 24 pulgadas,243,4.860000,"Monitor 24 pulgadas, Audífonos Bluetooth, Smar..."
4,medio_pago,6,Tarjeta crédito,1888,37.760000,"Tarjeta crédito, PSE, Tarjeta débito, Efectivo..."
5,calificacion_cliente,5,5.0,1198,42.362093,"5.0, 4.0, 3.0, 2.0, 1.0"


In [27]:
eda.resumen_categoricas(df_logistica)

,Variable,Valores únicos,Categoría más frecuente,Frecuencia,% de no nulos,Categorías
0,estado_evento,10,Listo para despacho,5006,10.012000,"Listo para despacho, Despachado, Orden enviada..."
1,centro_logistico,6,CEDI Bogotá,16491,33.041475,"CEDI Bogotá, CEDI Cali, CEDI Medellín, CEDI Ba..."
2,ciudad_destino,6,Bogotá D.C.,16510,33.049745,"Bogotá D.C., Cali, Medellín, Barranquilla, Buc..."
3,transportadora,5,Envia,4216,21.173162,"Envia, Coordinadora, Inter Rapidísimo, TCC, Se..."
4,incidencia,6,Error de clasificación,98,19.178082,"Error de clasificación, Cliente ausente, Noved..."
5,observacion,9,Validación realizada sin novedades.,412,23.911782,"Validación realizada sin novedades., Seguimien..."


In [28]:
eda.estadisticos_descriptivos(df_ventas, df_logistica)


############################################################
 PARTE 5: ESTADÍSTICOS DESCRIPTIVOS Y RESÚMENES
############################################################

--- 5.1 Variables numéricas de df_ventas ---
                  count         mean            std      min       p25   mediana       p75        max
cantidad         5000.0       1.4562       0.743903      1.0       1.0       1.0       2.0        4.0
precio_unitario  5000.0  455950.1600  445808.472395  75900.0  142400.0  289900.0  685900.0  2309900.0
descuento_pct    5000.0       5.2760       6.124649      0.0       0.0       5.0      10.0       20.0
valor_bruto      5000.0  668288.1200  828903.828696  75900.0  188800.0  391800.0  807400.0  8799600.0
valor_descuento  5000.0   34593.1040   79018.313185      0.0       0.0    5495.0   35970.0  1671920.0
valor_neto       5000.0  633695.0160  788075.292481  63920.0  176310.0  367110.0  764915.0  8799600.0

--- 5.2 Variables categóricas de df_ventas ---
            Variable 

## Parte 6. Preguntas de negocio (datos originales)

In [29]:
from src.eda.business_questions import responder_preguntas_negocio
responder_preguntas_negocio(df_ventas, df_logistica)


############################################################
 PARTE 6: 10 PREGUNTAS DE NEGOCIO
############################################################

1. ¿Cuál es el canal con mayor venta neta acumulada?
   R: Página web, con $1.286.922.005 (40,6% del total).
canal
Página web          1.286922e+09
Tienda física       9.552724e+08
Aplicación móvil    9.262807e+08
Name: valor_neto, dtype: float64

2. ¿Cuál es la categoría con mayor número de productos vendidos en Bogotá D.C.?
   R: Tecnología, con 650 unidades. Sin embargo, en valor neto lidera Deportes ($273.790.785): vende menos unidades, pero más caras.
                   unidades   valor_neto
categoria                               
Deportes                379  273790785.0
Electrodomésticos       507  145291150.0
Hogar                   437  157488760.0
Oficina                 385  216705250.0
Tecnología              650  240385895.0

3. ¿Qué tan recurrentes son los clientes?
   R: De 2.292 clientes, 1.458 (63,6%) compraron má

## Parte 7. Transformación (Medallion)

- **Bronze:** ya se guardó después de la extracción. Silver se construye **leyendo Bronze**, no desde la memoria.
- **Silver:** limpieza (`df_ventas_transformado`, `df_logisitica_transformado`), más `incidencias_calidad` y cuarentena (`rechazados/`).
- **Gold:** integración por `pedido_id`: `ventas_logistica_gold` (1 fila por pedido), `eventos_gold` (1 fila por evento, sin montos) y las tablas agregadas `kpi_mensual`, `desempeno_transportadora` y `ventas_ciudad_canal`.

La justificación de cada decisión de imputación está en la sección 5 del informe y en los comentarios de `src/transform/transform_medallion.py`.

In [30]:
from src.pipeline import transformar
df_ventas_transformado, df_logisitica_transformado, df_gold = transformar(manifiesto)

[2026-10-05 20:30:27,077] [INFO] - PARTE 7: Transformación Medallion desde Bronze (20261005_203025)...


[2026-10-05 20:30:27,131] [INFO] - [SILVER] Ventas: filas idénticas eliminadas = 0


[2026-10-05 20:30:27,216] [INFO] - [SILVER] Logística: filas idénticas eliminadas = 99


[2026-10-05 20:30:27,252] [INFO] - [SILVER] evento_id repetidos con contenido distinto resueltos: 1


[2026-10-05 20:30:28,253] [INFO] - [VALIDACIÓN] OK    Silver ventas: id_venta único - 0 repetidos


...

[2026-10-05 20:30:29,592] [INFO] - Guardado data\gold\kpi_mensual.parquet (6 filas x 14 columnas)


[2026-10-05 20:30:29,598] [INFO] - Guardado data\gold\desempeno_transportadora.parquet (5 filas x 9 columnas)


[2026-10-05 20:30:29,604] [INFO] - Guardado data\gold\ventas_ciudad_canal.parquet (18 filas x 8 columnas)


[2026-10-05 20:30:29,605] [INFO] - [GOLD] Eventos: (49900, 25) | Pedidos: (5000, 38) | Tablas agregadas: ['kpi_mensual', 'desempeno_transportadora', 'ventas_ciudad_canal']


[2026-10-05 20:30:29,611] [INFO] - Manifiesto de la ejecución: data/manifiestos\20261005_203025.json


In [31]:
pd.DataFrame({
    "bronze": [df_ventas.shape, df_logistica.shape],
    "silver": [df_ventas_transformado.shape, df_logisitica_transformado.shape],
}, index=["ventas", "logistica"])

,bronze,silver
ventas,"(5000, 17)","(5000, 17)"
logistica,"(50000, 14)","(49900, 13)"


In [32]:
pd.DataFrame({"nulos_bronze": df_logistica.isna().sum(),
              "nulos_silver": df_logisitica_transformado.isna().sum()})

,nulos_bronze,nulos_silver
Unnamed: 13,49999,NaN
centro_logistico,90,0.0
ciudad_destino,45,0.0
costo_envio,0,0.0
estado_evento,0,0.0
evento_id,0,0.0
fecha_evento,65,65.0
fecha_prometida_entrega,55,0.0
incidencia,49489,0.0
numero_guia,30068,29944.0


In [33]:
df_gold.head()

,id_venta,pedido_id,fecha_venta,id_cliente,ciudad,canal,id_tienda,categoria,producto,cantidad,precio_unitario,descuento_pct,valor_bruto,valor_descuento,valor_neto,medio_pago,calificacion_cliente,n_eventos,n_estados,estado_final,fecha_ultimo_evento,fecha_prometida_entrega,centro_logistico,ciudad_destino,transportadora,numero_guia,costo_envio,tiempo_total_horas,n_incidencias,trazabilidad_completa,guia_compartida,fecha_entrega,dias_retraso,entregado_tarde,entregado,dias_ciclo_entrega,dias_prometidos,mes_venta
0,1,PED000001,2026-01-18 22:06:15,CLI00011,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Banda caminadora,1,2199900.0,10.0,2199900.0,219990.0,1979910.0,Tarjeta crédito,<NA>,10,10,Entregado,2026-01-20 14:18:58,2026-01-20 14:06:15,CEDI Bogotá,Bogotá D.C.,TCC,KMS20283624,8900,40.20,0,True,False,2026-01-20 14:18:58,0.01,True,True,1.68,1.67,2026-01
1,2,PED000002,2026-04-25 21:07:01,CLI00802,Cali,Página web,NaN,Deportes,Mancuernas ajustables,1,399900.0,10.0,399900.0,39990.0,359910.0,Tarjeta crédito,<NA>,10,10,Entregado,2026-04-27 11:00:48,2026-04-27 21:07:01,CEDI Cali,Cali,Envia,KMS11084230,8900,37.89,0,True,False,2026-04-27 11:00:48,-0.42,False,True,1.58,2.00,2026-04
2,3,PED000003,2026-03-28 02:39:43,CLI00255,Medellín,Página web,NaN,Tecnología,Tablet 10 pulgadas,1,899900.0,10.0,899900.0,89990.0,809910.0,Tarjeta crédito,4,10,10,Entregado,2026-03-29 22:55:26,2026-03-30 00:39:43,CEDI Medellín,Medellín,Inter Rapidísimo,KMS39135059,13900,44.26,0,True,False,2026-03-29 22:55:26,-0.07,False,True,1.84,1.92,2026-03
3,4,PED000004,2026-04-30 12:35:40,CLI00383,Medellín,Aplicación móvil,NaN,Electrodomésticos,Freidora de aire,2,472400.0,15.0,944800.0,141720.0,803080.0,PSE,5,10,10,Entregado,2026-05-03 08:20:40,2026-05-02 10:35:40,CEDI Medellín,Medellín,Envia,KMS32561389,13900,67.75,0,True,False,2026-05-03 08:20:40,0.91,True,True,2.82,1.92,2026-04
4,5,PED000005,2026-04-04 00:18:12,CLI01286,Bogotá D.C.,Aplicación móvil,NaN,Deportes,Balón fútbol,1,119900.0,0.0,119900.0,0.0,119900.0,Tarjeta crédito,<NA>,10,10,Entregado,2026-04-05 08:43:56,2026-04-05 16:18:12,CEDI Bogotá,Bogotá D.C.,Coordinadora,KMS90585430,8900,32.43,0,True,False,2026-04-05 08:43:56,-0.32,False,True,1.35,1.67,2026-04


### Validaciones de calidad

El pipeline verifica estas reglas antes de guardar Silver y Gold; si alguna falla, se detiene sin publicar.

In [34]:
val = manifiesto.datos["validaciones"]
pd.DataFrame(val["silver"]["reglas"] + val["gold"]["reglas"])

,Capa,Regla,Cumple,Detalle
0,Silver ventas,id_venta único,True,0 repetidos
1,Silver ventas,pedido_id único,True,0 repetidos
2,Silver ventas,fecha_venta es datetime y sin nulos,True,0 nulos
3,Silver ventas,calificacion_cliente entre 1 y 5 (o nula),True,0 fuera de rango
4,Silver ventas,Montos no negativos,True,0 negativos
5,Silver ventas,Ciudades dentro de la cobertura,True,todas
6,Silver logística,evento_id único,True,0 repetidos
7,Silver logística,Sin filas duplicadas,True,0 duplicadas
8,Silver logística,Sin columnas residuales del Excel,True,
9,Silver logística,Fechas con tipo datetime,True,


### Incidencias de calidad y cuarentena

In [35]:
incidencias = pd.read_parquet("data/silver/incidencias_calidad.parquet")
incidencias.groupby(["tipo", "accion"]).size().rename("registros").sort_values(ascending=False)

tipo                                         accion                             
Fila duplicada                               Eliminada                              99
Trazabilidad incompleta                      Marcado en Gold                        98
centro_logistico nulo                        Recuperado del mismo pedido            89
transportadora nulo en estado de transporte  Completado con el valor del pedido     80
fecha_evento nula                            Se mantiene nulo (no recuperable)      65
numero_guia nulo en estado de transporte     Completado con el valor del pedido     60
fecha_prometida_entrega nulo                 Recuperado del mismo pedido            55
ciudad_destino nulo                          Recuperado del mismo pedido            45
Evento Entregado sin fecha                   Sin fecha de entrega en Gold            6
numero_guia compartido entre pedidos         Marcado en Gold                         2
Columna residual del Excel                   Elim

In [36]:
print("Ventas en cuarentena:", len(pd.read_parquet("data/silver/rechazados/ventas_rechazadas.parquet")),
      "| eventos en cuarentena:", len(pd.read_parquet("data/silver/rechazados/logistica_rechazada.parquet")))

Ventas en cuarentena: 0 | eventos en cuarentena: 0


### Tablas agregadas de Gold

In [37]:
pd.read_parquet("data/gold/kpi_mensual.parquet")

,mes_venta,pedidos,clientes,unidades,valor_bruto,valor_descuento,valor_neto,ticket_promedio,costo_envio_total,pct_entregas_tardias,dias_retraso_promedio,ciclo_mediano_dias,pct_pedidos_con_incidencia,calificacion_promedio
0,2026-01,830,714,1213,564702800.0,30983140.0,533719660.0,643035.73,11098790,17.27,-0.36,1.48,11.20,4.1
1,2026-02,764,673,1106,468124900.0,26685370.0,441439530.0,577800.43,10399770,17.59,-0.34,1.42,9.16,4.04
2,2026-03,867,732,1271,565496000.0,28940460.0,536555540.0,618864.52,12025490,17.57,-0.36,1.44,9.11,4.12
3,2026-04,851,724,1241,573028200.0,29040480.0,543987720.0,639233.51,11585180,16.73,-0.35,1.45,10.69,4.14
4,2026-05,821,710,1176,551932300.0,27204300.0,524728000.0,639132.76,11255760,16.28,-0.35,1.44,9.62,4.08
5,2026-06,867,729,1274,618156400.0,30111770.0,588044630.0,678252.17,11689440,15.99,-0.35,1.44,11.42,4.08


In [38]:
pd.read_parquet("data/gold/desempeno_transportadora.parquet")

,transportadora,pedidos,pct_entregas_tardias,retraso_promedio_tardias,ciclo_mediano_dias,tiempo_total_mediano_horas,costo_envio_promedio,pct_pedidos_con_incidencia,calificacion_promedio
0,TCC,957,17.78,0.43,1.44,34.50,13588.90,10.45,4.08
1,Inter Rapidísimo,1007,17.27,0.47,1.46,34.82,13721.29,10.23,4.1
2,Servientrega,950,16.68,0.42,1.44,34.46,13531.32,10.21,4.03
3,Envia,1059,16.41,0.42,1.46,34.86,13509.45,10.48,4.12
4,Coordinadora,1027,16.39,0.43,1.45,34.54,13701.31,9.74,4.13


In [39]:
pd.read_parquet("data/gold/ventas_ciudad_canal.parquet").head(6)

,ciudad,canal,pedidos,unidades,valor_neto,ticket_promedio,pct_entregas_tardias,pct_valor_neto_total
0,Bogotá D.C.,Página web,674,962,425646450.0,631522.92,16.07,13.43
1,Bogotá D.C.,Tienda física,497,735,311320580.0,626399.56,16.53,9.83
2,Bogotá D.C.,Aplicación móvil,482,661,296694810.0,615549.40,16.7,9.36
3,Cali,Página web,408,581,258843035.0,634419.20,15.97,8.17
4,Medellín,Página web,365,538,243556060.0,667276.88,18.96,7.69
5,Medellín,Aplicación móvil,297,437,188696895.0,635343.08,17.51,5.96


### Linaje y reproceso

Cada ejecución deja un manifiesto en `data/manifiestos/`. Con `python reprocesar.py --carga AAAAMMDD_HHMMSS` se reconstruyen Silver y Gold desde cualquier carga del histórico, sin consultar MySQL ni el Excel.

In [40]:
import json
print(json.dumps({k: manifiesto.datos[k] for k in ["id_ejecucion", "id_carga", "origen", "estado", "duracion_s", "fuentes"]},
                 ensure_ascii=False, indent=2))

{
  "id_ejecucion": "20261005_203025",
  "id_carga": "20261005_203025",
  "origen": "extraccion",
  "estado": "OK",
  "duracion_s": 4.6,
  "fuentes": {
    "ventas": {
      "tipo": "MySQL",
      "base": "clientes",
      "tabla": "ventas",
      "filas": 5000,
      "columnas": 17
    },
    "logistica": {
      "tipo": "Excel",
      "archivo": "data/kaismart_eventos_logisticos.xlsx",
      "sha256": "98c55cd5af70ba33703d5e1ca82611552fcf19f20bbcb3236d84d272cd054f0d",
      "filas": 50000,
      "columnas": 14
    }
  }
}


In [41]:
from src.load.load import listar_cargas
listar_cargas()

['20261005_203025']

In [42]:
print("Tiempo de ciclo mediano (días):", df_gold["dias_ciclo_entrega"].median(),
      "| prometido:", df_gold["dias_prometidos"].median())
df_gold.groupby("mes_venta")["entregado_tarde"].mean().mul(100).round(1)

Tiempo de ciclo mediano (días): 1.45 | prometido: 1.92


mes_venta
2026-01    17.3
2026-02    17.6
2026-03    17.6
2026-04    16.7
2026-05    16.3
2026-06    16.0
Name: entregado_tarde, dtype: Float64

In [43]:
print("Entregas tardías:", f"{df_gold['entregado_tarde'].mean():.1%}")
df_gold["dias_retraso"].describe()

Entregas tardías: 16.9%


count    4984.000000
mean       -0.353439
std         0.448270
min        -1.400000
25%        -0.650000
50%        -0.430000
75%        -0.160000
max         1.500000
Name: dias_retraso, dtype: float64

## Parte 8. Automatización con Schedule

El orquestador corre en un proceso aparte, porque es un ciclo infinito. Se ejecuta desde la raíz del proyecto:

```bash
python orchestrator.py
```

Hace una ejecución inmediata y después repite cada `orquestador.cada_horas` (definido en `config.yaml`). Aquí solo se muestra la tarea programada:

In [44]:
import schedule
from orchestrator import tarea_pipeline_etl
schedule.clear()
schedule.every(1).hours.do(tarea_pipeline_etl)
schedule.get_jobs()

[Every 1 hour do tarea_pipeline_etl() (last run: [never], next run: 2026-10-05 21:30:29)]

## Conclusiones

Las conclusiones (hallazgos de `df_ventas`, de `df_logistica` y de la transformación) están en la sección 7 de `docs/Informe_KaismartSolutions.html`. Ese informe se genera con `python generar_informe.py` a partir de las capas Medallion.